In [1]:
import pandas as pd

# Read in dataset from data preprocessing
df = pd.read_csv("../data/clean_matches.csv")
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s)
0,1,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez']
1,2,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M..."
2,3,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']"
3,4,2017-05-11,Old Trafford,Man United,1,Celta Vigo,1,"['Marounne Fellaini', 'Facundo Roncaglia']"
4,5,2017-12-14,Old Trafford,Man United,1,Bournemouth,0,['Romelu Lukaku']
5,6,2019-10-10,Old Trafford,Man United,3,Brighton,1,"['Andreas Pereira', 'Scott McTominay', 'Lewis ..."
6,7,2019-12-01,Old Trafford,Man United,2,Aston Villa,2,"['Jack Grealish', 'Tom Heaton (og)', 'Victor L..."
7,8,2019-12-14,King Power Stadium,Leicester,1,Norwich,1,"['Teemu Pukki', 'Tim Krul (og)']"
8,9,2020-02-24,Old Trafford,Man United,3,Watford,0,"['Bruno Fernandes', 'Antony Martial', 'Mason G..."
9,10,2021-12-11,Carrow Road,Norwich,0,Man United,1,['Cristiano Ronaldo']


In [2]:
# Function that builds the search query that will be input to Google
import calendar

def build_search_query(match):                              # Match will be row from the dataset
    # Set up variables that will be used in the string for search query
    home_team = match["home team"]
    away_team = match["away team"]
    home_score = match["home score"]
    away_score = match["away score"]

    month = calendar.month_name[int(match["date"][5:7])]    # Calendar converts the numbers 1-12 to the corresponding months January-December
    year = match["date"][0:4]
    return f"{home_team} {home_score} {away_team} {away_score} {month} {year} match report"  # Return the search query in an f-string

In [3]:
# Add the seach query as a column in the database
df["search query"] = df.apply(build_search_query, axis=1)
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s),search query
0,1,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez'],Man United 1 Birmingham 0 January 2008 match r...
1,2,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M...",Man United 4 Hull 3 November 2008 match report
2,3,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']",Aston Villa 2 Wycombe 0 January 2016 match report
3,4,2017-05-11,Old Trafford,Man United,1,Celta Vigo,1,"['Marounne Fellaini', 'Facundo Roncaglia']",Man United 1 Celta Vigo 1 May 2017 match report
4,5,2017-12-14,Old Trafford,Man United,1,Bournemouth,0,['Romelu Lukaku'],Man United 1 Bournemouth 0 December 2017 match...
5,6,2019-10-10,Old Trafford,Man United,3,Brighton,1,"['Andreas Pereira', 'Scott McTominay', 'Lewis ...",Man United 3 Brighton 1 October 2019 match report
6,7,2019-12-01,Old Trafford,Man United,2,Aston Villa,2,"['Jack Grealish', 'Tom Heaton (og)', 'Victor L...",Man United 2 Aston Villa 2 December 2019 match...
7,8,2019-12-14,King Power Stadium,Leicester,1,Norwich,1,"['Teemu Pukki', 'Tim Krul (og)']",Leicester 1 Norwich 1 December 2019 match report
8,9,2020-02-24,Old Trafford,Man United,3,Watford,0,"['Bruno Fernandes', 'Antony Martial', 'Mason G...",Man United 3 Watford 0 February 2020 match report
9,10,2021-12-11,Carrow Road,Norwich,0,Man United,1,['Cristiano Ronaldo'],Norwich 0 Man United 1 December 2021 match report


In [4]:
# API key needs to be kept private from public GITHUB repository
# Key stored in an .env file
# .env file listed within .gitignore
# This cell reads in the key and stores it with api_key

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("SERPAPI_KEY")

In [5]:
# Function that performs the Google search query
# Uses SerpAPI

from serpapi import GoogleSearch

def get_urls(match):
    # Parameters for the Google search
    params = {
        "q": match["search query"],                     # The string to be query                
        "api_key": api_key,             # The SerpAPI key
        "num": 20,                       # Max 20 links, but will naturally restrict to the first page of Google searches
        "google_domain": "google.co.uk",
        "location": "London, England, United Kingdom",
        "gl": "uk",
        "hl": "en"
    }

    search = GoogleSearch(params)       
    results = search.get_dict()         # Sends request to the API

    return [r["link"] for r in results.get("organic_results", [])]          # Extract links

In [7]:
# Test the function get_urls on the query generated for one of the matches
# Need to be careful about overusing this function: get 250 queries per month through free tier of SerpAPI

urls = get_urls(df.iloc[[1]])
urls

['https://www.espn.com/soccer/match/_/gameId/220076/birmingham-city-manchester-united',
 'https://www.premierleague.com/en/match/128642/manchester-united-vs-birmingham-city',
 'http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm',
 'https://www.skysports.com/football/manchester-united-vs-birmingham-city/teams/94538',
 'https://www.manutd.com/en/mutv/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2007-08',
 'https://www.mufcinfo.com/manupag/match_data/match_sql.php?my_match_date=2008-01-01',
 'https://www.manutd.com/en/mutv/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2009-10',
 'https://en.wikipedia.org/wiki/2006%E2%80%9307_Manchester_United_F.C._season']

In [ ]:
# Access the urls variable from the previous cell without needing to send another query to SERPAPI
urls

In [ ]:
# Able to redefine urls from above cell if I start a new session without needing to re-use a SERPAPI search
urls = ['https://www.youtube.com/watch?v=givC4ZvbqYc',
 'https://www.skysports.com/football/manchester-united-vs-birmingham-city/teams/94538',
 'https://www.transfermarkt.us/manchester-united_birmingham-city/index/spielbericht/81687',
 'https://www.manutd.com/en/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2007-08',
 'https://www.statmuse.com/fc/ask/full-time-scores-of-man-u-vs-birmingham-in-premiere-league-match-in-2008',
 'http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm',
 'https://www.premierleague.com/en/match/128642/manchester-united-vs-birmingham-city',
 'https://www.espn.ph/football/report/_/gameId/220076',
 'https://www.footballcritic.com/premier-league-manchester-united-fc-birmingham-city-fc/match-stats/42194']

In [ ]:
# Test on one match before expanding to all to preserve SerpAPI queries
# Remove this cell before roll-out

df = df.iloc[[1]].copy()        # Make a copy of the second entry in a sample dataframe
df["article urls"] = [urls]     # Add a new column of the list of url strings
df

In [36]:
df = df.iloc[[42,43]].copy()
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s),search query
42,43,2025-10-25,Old Trafford,Man United,4,Brighton,2,"['Matheus Cunha', 'Casemiro', 'Bryan Mbeumo (x...",Man United 4 Brighton 2 October 2025 match report
43,44,2025-12-15,Old Trafford,Man United,4,Bournemouth,4,"['Amad Diallo', 'Antoine Semenyo', 'Casemiro',...",Man United 4 Bournemouth 4 December 2025 match...


In [6]:
#df["search query"] = df.apply(build_search_query, axis=1)

df["article urls"] = df.apply(get_urls, axis=1)

In [7]:
df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s),search query,article urls
0,1,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez'],Man United 1 Birmingham 0 January 2008 match r...,[https://www.espn.co.uk/football/match/_/gameI...
1,2,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M...",Man United 4 Hull 3 November 2008 match report,[https://www.espn.co.uk/football/match/_/gameI...
2,3,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']",Aston Villa 2 Wycombe 0 January 2016 match report,[https://www.espn.co.uk/football/match/_/gameI...
3,4,2017-05-11,Old Trafford,Man United,1,Celta Vigo,1,"['Marounne Fellaini', 'Facundo Roncaglia']",Man United 1 Celta Vigo 1 May 2017 match report,[https://www.espn.co.uk/football/match/_/gameI...
4,5,2017-12-14,Old Trafford,Man United,1,Bournemouth,0,['Romelu Lukaku'],Man United 1 Bournemouth 0 December 2017 match...,[https://www.espn.co.uk/football/match/_/gameI...
5,6,2019-10-10,Old Trafford,Man United,3,Brighton,1,"['Andreas Pereira', 'Scott McTominay', 'Lewis ...",Man United 3 Brighton 1 October 2019 match report,[https://www.premierleague.com/en/match/987602...
6,7,2019-12-01,Old Trafford,Man United,2,Aston Villa,2,"['Jack Grealish', 'Tom Heaton (og)', 'Victor L...",Man United 2 Aston Villa 2 December 2019 match...,[https://www.manutd.com/en/news/match-report-u...
7,8,2019-12-14,King Power Stadium,Leicester,1,Norwich,1,"['Teemu Pukki', 'Tim Krul (og)']",Leicester 1 Norwich 1 December 2019 match report,[https://www.espn.co.uk/football/report/_/game...
8,9,2020-02-24,Old Trafford,Man United,3,Watford,0,"['Bruno Fernandes', 'Antony Martial', 'Mason G...",Man United 3 Watford 0 February 2020 match report,[https://www.espn.co.uk/football/match/_/gameI...
9,10,2021-12-11,Carrow Road,Norwich,0,Man United,1,['Cristiano Ronaldo'],Norwich 0 Man United 1 December 2021 match report,[https://www.espn.co.uk/football/match/_/gameI...


In [8]:
for row in df["article urls"]:
    for url in row:
        print(url)
    print("new match")

https://www.espn.co.uk/football/match/_/gameId/220076/birmingham-city-manchester-united
http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm
https://www.premierleague.com/en/match/128642/manchester-united-vs-birmingham-city
https://www.skysports.com/football/manchester-united-vs-birmingham-city/teams/94538
https://www.manutd.com/en/mutv/videos/detail/man-utd-1-birmingham-city-0-extended-highlights-premier-league-2007-08
https://www.mufcinfo.com/manupag/match_data/match_sql.php?my_match_date=2008-01-01
https://www.espn.co.uk/football/report/_/gameId/220076
https://www.facebook.com/groups/271475672143247/posts/709440608346749/
https://www.facebook.com/football70s80s/videos/birmingham-city-v-manchester-united-1977the-first-saturday-of-the-new-football-s/1747952412818369/
new match
https://www.espn.co.uk/football/match/_/gameId/242601/hull-city-manchester-united
http://news.bbc.co.uk/sport2/hi/football/eng_prem/7684745.stm
https://www.premierleague.com/en/match/266347
https://www.

In [9]:
# Read text from an articles

import requests
from bs4 import BeautifulSoup
import re

def extract_article(url):
    html = requests.get(url).text               # Gets the raw HTML code of the webpage
    soup = BeautifulSoup(html, "html.parser")   # Parses the HTML code into paragraphs/heading/etc. by looking for <p>/<h1>/etc.

    paragraphs = [p.get_text() for p in soup.find_all("p")]     # Gets all paragraph text
    text = "\n".join(paragraphs)
    text = re.sub(r"\s+", " ", text)             # Find all chunks of whitespace and replace them with a single space. \s is any whitespace character. \s+ means more than one in a row
    return text.strip()

# Testing the function

#extract_article("http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm")

In [10]:
import trafilatura

def extract_article(url):
    downloaded = trafilatura.fetch_url(url)
    text = trafilatura.extract(downloaded)
    return text

In [11]:
# Determine the source of an article from the url found by the Google search

from urllib.parse import urlparse

# A set of all trusted sources: aim is that found urls will include one of these within domain name
# NEEDS UPDATING TO INCLUDE MORE SOURCES
trusted_sources = {
    "bbc",
    "espn",
    "skysports",
    "guardian"
}
# trusted_sources = {
#     "bbc",
#     "skysports",
#     "espn",
#     "rte"
# }

def get_source(url):
    domain = urlparse(url).netloc.lower()               # netloc extracts the network location: i.e. "https://www.example.com:8080/page" goes to "www.example.com:8080"
    for name in trusted_sources:
        if name in domain:                              # Check if one of the trusted sources is in the domain name
            return name
    return "other"

# Testing the function
get_source("http://news.bbc.co.uk/sport2/hi/football/eng_prem/7163892.stm")

'bbc'

In [12]:
# Build article records into a Dataframe

# Create a list with an entry for each article from a trusted source for each match
article_records = []
for _, row in df.iterrows():                        # iterrows() iterates through tuples (*,*) of row index and corresponding one row Dataframe
    match_id = row["match_id"]
    urls = row["article urls"]
    for url in urls:
        source = get_source(url)

        # Add source, url and text for each trusted article for a given match
        if source in trusted_sources:
            text = extract_article(url)[:32750]   # Limit the number of characters in the text: Excel can only store up to approx 32750 characters before going to a new cell
            article_records.append({
                "match_id":match_id,
                "source":source,
                "url": url,
                "text": text
            })

# Convert data to a dataframe and store into a csv file
article_df = pd.DataFrame(article_records)
article_df.to_csv("../data/articles.csv", index=False)